In [23]:
import os
import numpy as np
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from PIL import Image
import random
import cv2
from scipy.ndimage import binary_dilation, gaussian_filter, distance_transform_edt
import matplotlib.pyplot as plt
from collections import defaultdict

# PATHS
SKELETON_DIR = r"C:\Users\jhoffm72\Documents\FilPHANGS\Data\ngc4321_F770W\Composites - Copy"
SOURCE_IMAGE = r"C:\Users\jhoffm72\Documents\FilPHANGS\Data\OriginalImages\ngc4321_F770W_JWST_Emission_starsub.fits"
TARGET_IMAGE = r"C:\Users\jhoffm72\Documents\FilPHANGS\Data\OriginalImages\ngc4535_F770W_JWST_Emission_starsub.fits"
OUTPUT_DIR   = r"C:\Users\jhoffm72\Documents\FilPHANGS\Output"

# PARAMETERS
N_BRIGHTEST_FILAMENTS = 10      # Number of brightest filaments to extract per skeleton
PATCH_RADIUS = 8                 # Radius around each filament pixel
MIN_FILAMENT_PIXELS = 20         # Minimum pixels for valid filament
MAX_PLACEMENT_ATTEMPTS = 500     # Attempts to place each patch
BORDER_WIDTH = 3                 # Width of rectangular border around inserted patches
MIN_SCALE = 0.8                  # Scaling range for inserted patches
MAX_SCALE = 1.2

# BLENDING PARAMETERS
BLENDING_METHOD = "feather"      # Options: "none", "gaussian", "feather", "poisson"
FEATHER_WIDTH = 5                # Width of feathering/blending zone (pixels)
GAUSSIAN_SIGMA = 2.0             # Sigma for Gaussian blending
BRIGHTNESS_MATCH = True          # Match brightness at edges to local background

# HELPER FUNCTIONS

def identify_connected_components(skeleton_img):
    """Identify connected components (individual filaments) in skeleton image."""
    binary = (skeleton_img > 0).astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    return labels, stats, num_labels


def get_brightest_filaments(skeleton_img, labels, stats, num_labels, n_brightest=10, min_pixels=20):
    """Find N brightest filaments based on average intensity of non-zero pixels."""
    filament_data = []
    
    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area < min_pixels:
            continue
        
        mask = (labels == label_id)
        filament_pixels = skeleton_img[mask]
        non_zero_pixels = filament_pixels[filament_pixels > 0]
        
        if len(non_zero_pixels) == 0:
            continue
            
        avg_brightness = np.mean(non_zero_pixels)
        coords = np.argwhere(mask)
        
        filament_data.append({
            'label_id': label_id,
            'avg_brightness': avg_brightness,
            'area': area,
            'coords': coords,
            'stats': stats[label_id]
        })
    
    filament_data.sort(key=lambda x: x['avg_brightness'], reverse=True)
    return filament_data


def extract_patch_around_filament(source_data, filament_coords, radius=8):
    """Extract patch from source image around filament coordinates."""
    h, w = source_data.shape
    
    row_coords = filament_coords[:, 0]
    col_coords = filament_coords[:, 1]
    
    row_min = max(0, row_coords.min() - radius)
    row_max = min(h, row_coords.max() + radius + 1)
    col_min = max(0, col_coords.min() - radius)
    col_max = min(w, col_coords.max() + radius + 1)
    
    patch_data = source_data[row_min:row_max, col_min:col_max].copy()
    patch_mask = np.zeros_like(patch_data, dtype=bool)
    
    shifted_coords = filament_coords.copy()
    shifted_coords[:, 0] -= row_min
    shifted_coords[:, 1] -= col_min
    
    for y, x in shifted_coords:
        y_min = max(0, y - radius)
        y_max = min(patch_data.shape[0], y + radius + 1)
        x_min = max(0, x - radius)
        x_max = min(patch_data.shape[1], x + radius + 1)
        
        yy, xx = np.ogrid[y_min:y_max, x_min:x_max]
        circle = (yy - y)**2 + (xx - x)**2 <= radius**2
        patch_mask[y_min:y_max, x_min:x_max] |= circle
    
    bbox = (row_min, row_max, col_min, col_max)
    return patch_data, patch_mask, bbox


def create_feather_mask(mask, feather_width=5):
    """
    Create a smooth feathering mask using distance transform.
    Values go from 0 (outside) to 1 (inside core) with smooth transition.
    """
    # Distance from edge (inside the mask)
    dist_inside = distance_transform_edt(mask)
    
    # Create smooth transition zone
    feather_mask = np.clip(dist_inside / feather_width, 0, 1)
    
    return feather_mask


def create_gaussian_blend_mask(mask, sigma=2.0):
    """
    Create smooth blending mask using Gaussian smoothing.
    """
    # Convert mask to float
    float_mask = mask.astype(np.float32)
    
    # Apply Gaussian smoothing
    smooth_mask = gaussian_filter(float_mask, sigma=sigma)
    
    return smooth_mask


def match_brightness_at_boundary(patch_data, target_region, mask, boundary_width=3):
    """
    Adjust patch brightness to match target image at the boundary.
    """
    # Create boundary region (dilated mask minus original mask)
    boundary = binary_dilation(mask, iterations=boundary_width) & ~mask
    
    if not boundary.any():
        return patch_data
    
    # Get brightness in boundary regions
    patch_boundary = patch_data[boundary & (patch_data > 0)]
    target_boundary = target_region[boundary & (target_region > 0)]
    
    if len(patch_boundary) == 0 or len(target_boundary) == 0:
        return patch_data
    
    # Calculate brightness ratio
    patch_mean = np.median(patch_boundary)
    target_mean = np.median(target_boundary)
    
    if patch_mean > 0:
        ratio = target_mean / patch_mean
        # Apply scaling with limits to avoid extreme values
        ratio = np.clip(ratio, 0.5, 2.0)
        adjusted_patch = patch_data * ratio
        return adjusted_patch
    
    return patch_data


def insert_patch_with_blending(canvas, patch_data, patch_mask, row, col, 
                                method="feather", feather_width=5, 
                                gaussian_sigma=2.0, match_brightness=True):
    """
    Insert patch with various blending methods.
    
    Methods:
    - "none": Hard edge insertion (original method)
    - "gaussian": Gaussian smoothing at edges
    - "feather": Distance-based feathering
    - "poisson": Poisson blending (more complex, experimental)
    """
    ph, pw = patch_data.shape
    target_region = canvas[row:row+ph, col:col+pw].copy()
    
    # Match brightness at boundary if requested
    if match_brightness:
        patch_data = match_brightness_at_boundary(patch_data, target_region, patch_mask)
    
    if method == "none":
        # Original hard-edge method
        canvas[row:row+ph, col:col+pw][patch_mask] = patch_data[patch_mask]
    
    elif method == "gaussian":
        # Gaussian blending
        blend_mask = create_gaussian_blend_mask(patch_mask, sigma=gaussian_sigma)
        
        # Blend: target * (1 - mask) + patch * mask
        blended = target_region * (1 - blend_mask) + patch_data * blend_mask
        canvas[row:row+ph, col:col+pw] = blended
    
    elif method == "feather":
        # Feathering with distance transform
        feather_mask = create_feather_mask(patch_mask, feather_width=feather_width)
        
        # Blend with feathering
        blended = target_region * (1 - feather_mask) + patch_data * feather_mask
        canvas[row:row+ph, col:col+pw] = blended
    
    elif method == "poisson":
        # Poisson blending (seamless cloning)
        # This is more complex and may not work well for all cases
        try:
            # Only blend where mask is True
            if patch_mask.sum() > 0:
                # Create a center point for the patch
                center = (col + pw // 2, row + ph // 2)
                
                # Convert to uint8 for cv2.seamlessClone
                patch_8bit = np.clip(patch_data / np.nanmax(patch_data) * 255, 0, 255).astype(np.uint8)
                canvas_8bit = np.clip(canvas / np.nanmax(canvas) * 255, 0, 255).astype(np.uint8)
                mask_8bit = (patch_mask * 255).astype(np.uint8)
                
                # Apply seamless cloning to region
                blended_8bit = cv2.seamlessClone(
                    patch_8bit, canvas_8bit[row:row+ph, col:col+pw], 
                    mask_8bit, (pw//2, ph//2), cv2.NORMAL_CLONE
                )
                
                # Convert back to original scale
                scale_factor = np.nanmax(canvas) / 255.0
                canvas[row:row+ph, col:col+pw] = blended_8bit.astype(np.float32) * scale_factor
            else:
                # Fallback to feathering
                feather_mask = create_feather_mask(patch_mask, feather_width=feather_width)
                blended = target_region * (1 - feather_mask) + patch_data * feather_mask
                canvas[row:row+ph, col:col+pw] = blended
        except Exception as e:
            print(f"    Warning: Poisson blending failed, using feather method. Error: {e}")
            feather_mask = create_feather_mask(patch_mask, feather_width=feather_width)
            blended = target_region * (1 - feather_mask) + patch_data * feather_mask
            canvas[row:row+ph, col:col+pw] = blended


def check_bbox_overlap(bbox1, bbox2):
    """Check if two bounding boxes overlap."""
    r1_min, r1_max, c1_min, c1_max = bbox1
    r2_min, r2_max, c2_min, c2_max = bbox2
    
    no_overlap = (r1_max <= r2_min or r2_max <= r1_min or 
                  c1_max <= c2_min or c2_max <= c1_min)
    
    return not no_overlap


def build_target_mask(data):
    """Build mask for valid placement regions in target image."""
    threshold = 10**-20
    mask_copy = (data > threshold).astype(np.uint8) * 255
    
    kernel_1 = np.ones((5, 5), np.uint8)
    dilated = cv2.dilate(mask_copy, kernel_1, iterations=2)
    
    kernel_2 = np.ones((10, 10), np.uint8)
    eroded = cv2.erode(dilated, kernel_2, iterations=2)
    
    return eroded == 255


def can_place_patch(target_mask, occupied, row, col, patch_h, patch_w):
    """Check if patch can be placed at given location."""
    h, w = target_mask.shape
    
    if row + patch_h > h or col + patch_w > w:
        return False
    
    region_mask = target_mask[row:row+patch_h, col:col+patch_w]
    region_occupied = occupied[row:row+patch_h, col:col+patch_w]
    
    return region_mask.all() and not region_occupied.any()


def draw_rectangle_border(image, row, col, height, width, border_width, color):
    """Draw a rectangular border on an image."""
    h, w = image.shape[:2]
    
    row = max(0, row)
    col = max(0, col)
    row_end = min(h, row + height)
    col_end = min(w, col + width)
    
    image[row:min(row+border_width, row_end), col:col_end] = color
    image[max(row_end-border_width, row):row_end, col:col_end] = color
    image[row:row_end, col:min(col+border_width, col_end)] = color
    image[row:row_end, max(col_end-border_width, col):col_end] = color


# MAIN PROCESSING

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 80)
print("FILAMENT INSERTION WITH ADVANCED BLENDING")
print("=" * 80)
print(f"Blending method: {BLENDING_METHOD}")
print(f"Feather width: {FEATHER_WIDTH}")
print(f"Gaussian sigma: {GAUSSIAN_SIGMA}")
print(f"Brightness matching: {BRIGHTNESS_MATCH}")
print("=" * 80)

print("\nSTEP 1: Load source image (where filaments will be extracted from)")
print("=" * 80)

with fits.open(SOURCE_IMAGE) as hdul:
    source_data = hdul[0].data.astype(np.float32)
    source_header = hdul[0].header.copy()
    if source_data.ndim == 3:
        source_data = source_data[0]

source_h, source_w = source_data.shape
print(f"Source image shape: {source_data.shape}")

source_occupied = np.zeros((source_h, source_w), dtype=bool)

print("\n" + "=" * 80)
print("STEP 2: Load target image (where filaments will be inserted)")
print("=" * 80)

with fits.open(TARGET_IMAGE) as hdul:
    target_header = hdul[0].header.copy()
    target_data = hdul[0].data.astype(np.float32)
    if target_data.ndim == 3:
        target_data = target_data[0]

canvas = target_data.copy()
target_h, target_w = canvas.shape
print(f"Target image shape: {canvas.shape}")

print("\nBuilding target mask...")
target_mask = build_target_mask(target_data)
print(f"Valid pixels: {target_mask.sum():,} ({100*target_mask.mean():.1f}%)")

occupied = np.zeros((target_h, target_w), dtype=bool)

print("\n" + "=" * 80)
print("STEP 3: Process skeleton composites (with non-overlapping extraction)")
print("=" * 80)

skeleton_files = [f for f in os.listdir(SKELETON_DIR) if f.lower().endswith('.fits')]
print(f"Found {len(skeleton_files)} skeleton FITS files\n")

all_patches = []
source_patch_locations = []

for skel_file in skeleton_files:
    skel_path = os.path.join(SKELETON_DIR, skel_file)
    print(f"\n--- Processing: {skel_file} ---")
    
    with fits.open(skel_path) as hdul:
        skeleton_data = hdul[0].data.astype(np.float32)
        if skeleton_data.ndim == 3:
            skeleton_data = skeleton_data[0]
    
    labels, stats, num_labels = identify_connected_components(skeleton_data)
    print(f"  Found {num_labels - 1} filaments (excluding background)")
    
    all_filaments = get_brightest_filaments(
        skeleton_data, labels, stats, num_labels,
        n_brightest=999999,
        min_pixels=MIN_FILAMENT_PIXELS
    )
    
    print(f"  Candidate filaments: {len(all_filaments)}")
    
    selected_filaments = []
    selected_bboxes = []
    
    for fil in all_filaments:
        if len(selected_filaments) >= N_BRIGHTEST_FILAMENTS:
            break
        
        row_coords = fil['coords'][:, 0]
        col_coords = fil['coords'][:, 1]
        
        row_min = max(0, row_coords.min() - PATCH_RADIUS)
        row_max = min(source_h, row_coords.max() + PATCH_RADIUS + 1)
        col_min = max(0, col_coords.min() - PATCH_RADIUS)
        col_max = min(source_w, col_coords.max() + PATCH_RADIUS + 1)
        
        candidate_bbox = (row_min, row_max, col_min, col_max)
        
        overlaps = False
        for existing_bbox in selected_bboxes:
            if check_bbox_overlap(candidate_bbox, existing_bbox):
                overlaps = True
                break
        
        if not overlaps:
            selected_filaments.append(fil)
            selected_bboxes.append(candidate_bbox)
            print(f"     Selected Label {fil['label_id']}: "
                  f"avg_brightness={fil['avg_brightness']:.2e}, "
                  f"area={fil['area']} px")
        else:
            print(f"     Skipped Label {fil['label_id']}: overlaps")
    
    print(f"  Final selection: {len(selected_filaments)} non-overlapping filaments")
    
    for fil in selected_filaments:
        patch_data, patch_mask, bbox = extract_patch_around_filament(
            source_data, fil['coords'], radius=PATCH_RADIUS
        )
        
        patch_data_masked = patch_data.copy()
        patch_data_masked[~patch_mask] = 0
        
        row_min, row_max, col_min, col_max = bbox
        source_occupied[row_min:row_max, col_min:col_max] = True
        
        source_patch_locations.append({
            'row': row_min,
            'col': col_min,
            'height': row_max - row_min,
            'width': col_max - col_min,
            'source_file': skel_file,
            'label_id': fil['label_id']
        })
        
        all_patches.append({
            'data': patch_data_masked,
            'mask': patch_mask,
            'bbox': bbox,
            'source_file': skel_file,
            'label_id': fil['label_id'],
            'avg_brightness': fil['avg_brightness']
        })

print(f"\n{'=' * 80}")
print(f"STEP 4: Insert {len(all_patches)} patches with {BLENDING_METHOD} blending")
print("=" * 80)

inserted_patches = []

for i, patch_info in enumerate(all_patches, 1):
    patch_data = patch_info['data']
    patch_mask = patch_info['mask']
    ph, pw = patch_data.shape
    
    scale = random.uniform(MIN_SCALE, MAX_SCALE)
    if scale != 1.0:
        new_h = max(4, int(ph * scale))
        new_w = max(4, int(pw * scale))
        patch_data = cv2.resize(patch_data, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        patch_mask = cv2.resize(patch_mask.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        ph, pw = new_h, new_w
    
    placed = False
    for attempt in range(MAX_PLACEMENT_ATTEMPTS):
        row = random.randint(0, target_h - ph - 1)
        col = random.randint(0, target_w - pw - 1)
        
        if can_place_patch(target_mask, occupied, row, col, ph, pw):
            # Insert with blending
            insert_patch_with_blending(
                canvas, patch_data, patch_mask, row, col,
                method=BLENDING_METHOD,
                feather_width=FEATHER_WIDTH,
                gaussian_sigma=GAUSSIAN_SIGMA,
                match_brightness=BRIGHTNESS_MATCH
            )
            
            occupied[row:row+ph, col:col+pw] = True
            
            inserted_patches.append({
                'row': row,
                'col': col,
                'height': ph,
                'width': pw,
                'mask': patch_mask.copy(),
                **patch_info
            })
            
            placed = True
            print(f"  [{i}/{len(all_patches)}] Inserted patch from {patch_info['source_file']} "
                  f"(label {patch_info['label_id']}) at ({row}, {col}) size ({ph}x{pw}) "
                  f"[attempt {attempt + 1}]")
            break
    
    if not placed:
        print(f"  [{i}/{len(all_patches)}] FAILED to place patch from {patch_info['source_file']} "
              f"(label {patch_info['label_id']}) after {MAX_PLACEMENT_ATTEMPTS} attempts")

print(f"\nSuccessfully inserted {len(inserted_patches)} / {len(all_patches)} patches")

print("\n" + "=" * 80)
print("STEP 5: Create boxed source image (showing extracted patches)")
print("=" * 80)

source_boxed_fits = source_data.copy()
border_value = np.nanmax(source_data) * 2

for loc in source_patch_locations:
    draw_rectangle_border(
        source_boxed_fits,
        loc['row'],
        loc['col'],
        loc['height'],
        loc['width'],
        BORDER_WIDTH,
        border_value
    )

out_source_fits = os.path.join(OUTPUT_DIR, "source_with_boxes.fits")
fits.PrimaryHDU(data=source_boxed_fits, header=source_header).writeto(out_source_fits, overwrite=True)
print(f"Saved boxed source FITS: {out_source_fits}")

zscale = ZScaleInterval()
vmin, vmax = zscale.get_limits(source_data)
stretched = np.clip((source_data - vmin) / (vmax - vmin), 0, 1)
rgb_source = np.stack([(stretched * 255).astype(np.uint8)] * 3, axis=-1)

for loc in source_patch_locations:
    draw_rectangle_border(
        rgb_source,
        loc['row'],
        loc['col'],
        loc['height'],
        loc['width'],
        BORDER_WIDTH,
        [255, 0, 0]
    )

out_source_png = os.path.join(OUTPUT_DIR, "source_with_boxes.png")
Image.fromarray(rgb_source, mode="RGB").save(out_source_png)
print(f"Saved boxed source PNG: {out_source_png}")

print("\n" + "=" * 80)
print("STEP 6: Save target image outputs")
print("=" * 80)

out_fits_plain = os.path.join(OUTPUT_DIR, "target_with_inserts.fits")
fits.PrimaryHDU(data=canvas, header=target_header).writeto(out_fits_plain, overwrite=True)
print(f"Saved plain target FITS: {out_fits_plain}")

canvas_highlighted = canvas.copy()
border_value = np.nanmax(canvas) * 2

for patch in inserted_patches:
    draw_rectangle_border(
        canvas_highlighted, 
        patch['row'], 
        patch['col'], 
        patch['height'], 
        patch['width'],
        BORDER_WIDTH,
        border_value
    )

out_fits_highlighted = os.path.join(OUTPUT_DIR, "target_with_inserts_highlighted.fits")
fits.PrimaryHDU(data=canvas_highlighted, header=target_header).writeto(out_fits_highlighted, overwrite=True)
print(f"Saved highlighted target FITS: {out_fits_highlighted}")

vmin, vmax = zscale.get_limits(canvas)
stretched = np.clip((canvas - vmin) / (vmax - vmin), 0, 1)
rgb = np.stack([(stretched * 255).astype(np.uint8)] * 3, axis=-1)

for patch in inserted_patches:
    draw_rectangle_border(
        rgb,
        patch['row'],
        patch['col'],
        patch['height'],
        patch['width'],
        BORDER_WIDTH,
        [255, 0, 0]
    )

out_png = os.path.join(OUTPUT_DIR, "target_with_inserts_highlighted.png")
Image.fromarray(rgb, mode="RGB").save(out_png)
print(f"Saved highlighted target PNG: {out_png}")

stretched_plain = np.clip((canvas - vmin) / (vmax - vmin), 0, 1)
rgb_plain = np.stack([(stretched_plain * 255).astype(np.uint8)] * 3, axis=-1)
out_png_plain = os.path.join(OUTPUT_DIR, "target_with_inserts_plain.png")
Image.fromarray(rgb_plain, mode="RGB").save(out_png_plain)
print(f"Saved plain target PNG: {out_png_plain}")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Blending method used: {BLENDING_METHOD}")
print(f"Total patches processed: {len(all_patches)}")
print(f"Successfully inserted into target: {len(inserted_patches)}")
print(f"Patches boxed in source image: {len(source_patch_locations)}")
print(f"\nOutput files:")
print(f"  SOURCE IMAGE:")
print(f"    - {out_source_fits}")
print(f"    - {out_source_png}")
print(f"  TARGET IMAGE:")
print(f"    - {out_fits_plain}")
print(f"    - {out_fits_highlighted}")
print(f"    - {out_png_plain}")
print(f"    - {out_png}")
print("=" * 80)
print("DONE!")
print("=" * 80)


FILAMENT INSERTION WITH ADVANCED BLENDING
Blending method: feather
Feather width: 5
Gaussian sigma: 2.0
Brightness matching: True

STEP 1: Load source image (where filaments will be extracted from)
Source image shape: (2391, 2548)

STEP 2: Load target image (where filaments will be inserted)
Target image shape: (2062, 1679)

Building target mask...
Valid pixels: 1,736,127 (50.1%)

STEP 3: Process skeleton composites (with non-overlapping extraction)
Found 3 skeleton FITS files


--- Processing: ngc4321_F770W_JWST_Emission_starsub_CDDss0016pc.fits_Composites.fits ---
  Found 1626 filaments (excluding background)
  Candidate filaments: 1626
    ✓ Selected Label 453: avg_brightness=2.55e+02, area=31 px
    ✓ Selected Label 770: avg_brightness=2.55e+02, area=39 px
    ✓ Selected Label 988: avg_brightness=2.55e+02, area=39 px
    ✓ Selected Label 1132: avg_brightness=2.55e+02, area=40 px
    ✓ Selected Label 1347: avg_brightness=2.55e+02, area=31 px
    ✓ Selected Label 775: avg_brightness=

C:\Users\jhoffm72\AppData\Local\Temp\ipykernel_3828\569656534.py:521: RuntimeWarning: invalid value encountered in cast
  rgb_source = np.stack([(stretched * 255).astype(np.uint8)] * 3, axis=-1)


Saved boxed source PNG: C:\Users\jhoffm72\Documents\FilPHANGS\Output\source_with_boxes.png

STEP 6: Save target image outputs
Saved plain target FITS: C:\Users\jhoffm72\Documents\FilPHANGS\Output\target_with_inserts.fits
Saved highlighted target FITS: C:\Users\jhoffm72\Documents\FilPHANGS\Output\target_with_inserts_highlighted.fits
Saved highlighted target PNG: C:\Users\jhoffm72\Documents\FilPHANGS\Output\target_with_inserts_highlighted.png


C:\Users\jhoffm72\AppData\Local\Temp\ipykernel_3828\569656534.py:566: RuntimeWarning: invalid value encountered in cast
  rgb = np.stack([(stretched * 255).astype(np.uint8)] * 3, axis=-1)
C:\Users\jhoffm72\AppData\Local\Temp\ipykernel_3828\569656534.py:584: RuntimeWarning: invalid value encountered in cast
  rgb_plain = np.stack([(stretched_plain * 255).astype(np.uint8)] * 3, axis=-1)


Saved plain target PNG: C:\Users\jhoffm72\Documents\FilPHANGS\Output\target_with_inserts_plain.png

SUMMARY
Blending method used: feather
Total patches processed: 30
Successfully inserted into target: 30
Patches boxed in source image: 30

Output files:
  SOURCE IMAGE:
    - C:\Users\jhoffm72\Documents\FilPHANGS\Output\source_with_boxes.fits
    - C:\Users\jhoffm72\Documents\FilPHANGS\Output\source_with_boxes.png
  TARGET IMAGE:
    - C:\Users\jhoffm72\Documents\FilPHANGS\Output\target_with_inserts.fits
    - C:\Users\jhoffm72\Documents\FilPHANGS\Output\target_with_inserts_highlighted.fits
    - C:\Users\jhoffm72\Documents\FilPHANGS\Output\target_with_inserts_plain.png
    - C:\Users\jhoffm72\Documents\FilPHANGS\Output\target_with_inserts_highlighted.png
DONE!
